# Loading and Exploring Data

Real data science starts with loading data. Pandas supports dozens of file formats, but CSV is by far the most common. Knowing the key parameters of `read_csv` and establishing a solid exploratory workflow will save you hours of debugging.

## Learning Objectives

- Master `pd.read_csv` with key parameters (sep, header, dtype, na_values, etc.)
- Know when to use `read_json`, `read_excel`, and `read_parquet`
- Save data efficiently with `to_csv` and `to_parquet`
- Develop a systematic exploratory data analysis workflow
- Use `value_counts` for categorical analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## pd.read_csv: The Workhorse

CSV (Comma-Separated Values) is the most common data format. `read_csv` has many parameters—knowing the important ones saves debugging time.

In [ ]:
# Basic read
df = pd.read_csv('../../data/titanic.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Key parameters:

# sep: delimiter (default is comma)
# df = pd.read_csv('file.tsv', sep='\t')  # Tab-separated
# df = pd.read_csv('file.txt', sep=';')   # Semicolon-separated

# header: which row contains column names (default 0, None if no header)
# df = pd.read_csv('file.csv', header=None)  # No header row

# index_col: which column to use as index
df_indexed = pd.read_csv('../../data/titanic.csv', index_col='PassengerId')
print("With PassengerId as index:")
df_indexed.head()

In [ ]:
# usecols: only load specific columns (faster for large files)
df_small = pd.read_csv('../../data/titanic.csv', usecols=['Name', 'Age', 'Fare'])
print(f"Loaded columns: {df_small.columns.tolist()}")
df_small.head()

In [ ]:
# dtype: specify column types (prevents wrong inference)
df_typed = pd.read_csv('../../data/titanic.csv', dtype={'Pclass': str, 'Age': float})
print(f"Pclass dtype: {df_typed['Pclass'].dtype}")

In [ ]:
# na_values: additional values to treat as NaN
# df = pd.read_csv('file.csv', na_values=['N/A', 'missing', '-'])

# parse_dates: convert columns to datetime
# df = pd.read_csv('file.csv', parse_dates=['date_column'])

# nrows: only read first n rows (great for testing)
df_sample = pd.read_csv('../../data/titanic.csv', nrows=100)
print(f"Sampled rows: {len(df_sample)}")

## Other File Formats

In [ ]:
# JSON (common for API responses)
# df = pd.read_json('data.json')
# df = pd.read_json('data.json', lines=True)  # JSON Lines format

# Excel
# df = pd.read_excel('data.xlsx', sheet_name='Sheet1')

# Parquet (columnar format, fast and compact)
# df = pd.read_parquet('data.parquet')

print("Supported formats include: CSV, JSON, Excel, Parquet, SQL, HTML, and more")

## Saving Data: to_csv and to_parquet

In [ ]:
# Save to CSV
# df.to_csv('output.csv', index=False)  # index=False to not save row numbers

# Save to Parquet (PREFERRED for production)
# df.to_parquet('output.parquet')

# Why Parquet?
# 1. Much smaller file size (compressed)
# 2. Much faster to read/write
# 3. Preserves dtypes (no re-inference needed)
# 4. Supports column-wise reading (only load columns you need)

print("Parquet advantages:")
print("- 2-10x smaller than CSV")
print("- 5-20x faster to load")
print("- Preserves data types")

## Exploratory Data Analysis Workflow

Every time you load a new dataset, follow this systematic workflow:

1. **Shape**: How many rows and columns?
2. **Dtypes**: What are the data types?
3. **Nulls**: How much missing data?
4. **Value counts**: What are the distributions?
5. **Describe**: What are the statistics?

In [ ]:
# Load fresh data
df = pd.read_csv('../../data/titanic.csv')

# STEP 1: Shape
print("=" * 50)
print("STEP 1: Shape")
print("=" * 50)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

In [ ]:
# STEP 2: Data Types
print("=" * 50)
print("STEP 2: Data Types")
print("=" * 50)
print(df.dtypes)

In [ ]:
# STEP 3: Missing Values
print("=" * 50)
print("STEP 3: Missing Values")
print("=" * 50)
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(1)
null_df = pd.DataFrame({'Missing': null_counts, 'Percent': null_pct})
print(null_df[null_df['Missing'] > 0].sort_values('Missing', ascending=False))

In [ ]:
# STEP 4: Value Counts for Categorical Columns
print("=" * 50)
print("STEP 4: Categorical Distributions")
print("=" * 50)

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols += ['Survived', 'Pclass']  # Add discrete numeric cols

for col in categorical_cols[:4]:  # First 4
    print(f"\n{col}:")
    print(df[col].value_counts())

In [ ]:
# STEP 5: Statistical Summary
print("=" * 50)
print("STEP 5: Statistical Summary")
print("=" * 50)
df.describe()

## value_counts Deep Dive

`value_counts` is one of the most useful methods for understanding your data.

In [ ]:
# Basic value_counts
print("Pclass distribution:")
print(df['Pclass'].value_counts())

In [ ]:
# As percentages
print("Pclass distribution (percentages):")
print(df['Pclass'].value_counts(normalize=True).round(3))

In [ ]:
# Sort by index instead of counts
print("Pclass sorted by class number:")
print(df['Pclass'].value_counts().sort_index())

In [ ]:
# Include NaN values
print("Cabin values (including NaN):")
print(df['Cabin'].value_counts(dropna=False).head(10))

In [ ]:
# Binning continuous data with value_counts
print("Age distribution (binned):")
print(pd.cut(df['Age'], bins=[0, 18, 35, 50, 100]).value_counts().sort_index())

## Visualizing Categorical Data

In [ ]:
# Bar plots with value_counts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pclass
df['Pclass'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Passenger Class Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Sex
df['Sex'].value_counts().plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'])
axes[1].set_title('Gender Distribution')
axes[1].set_xlabel('Sex')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Survival rate by different factors
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# By Pclass
df.groupby('Pclass')['Survived'].mean().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Survival Rate by Class')
axes[0].set_ylabel('Survival Rate')
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_ylim(0, 1)

# By Sex
df.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[1], color=['#3498db', '#e74c3c'])
axes[1].set_title('Survival Rate by Sex')
axes[1].set_ylabel('Survival Rate')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylim(0, 1)

# By Embarked
df.groupby('Embarked')['Survived'].mean().plot(kind='bar', ax=axes[2], color='steelblue')
axes[2].set_title('Survival Rate by Embarkation')
axes[2].set_ylabel('Survival Rate')
axes[2].tick_params(axis='x', rotation=0)
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Exercises

### Exercise 1: Loading Data with Parameters

Load the Titanic dataset but:
1. Only load the columns: Name, Sex, Age, Fare, Survived
2. Use PassengerId as the index
3. Only load the first 200 rows

In [ ]:
# Load with the specified parameters
# YOUR CODE HERE

### Exercise 2: Exploratory Analysis

Using the full Titanic dataset, answer:
1. What percentage of passengers survived?
2. What is the most common embarkation port?
3. What is the median fare?

In [ ]:
# Answer the questions
# YOUR CODE HERE

### Exercise 3: Value Counts Analysis

For the Titanic dataset:
1. Find the distribution of ages binned into: 0-18, 18-30, 30-50, 50+
2. Find the percentage of passengers in each class
3. Create a bar plot of embarkation port distribution

In [ ]:
# Perform the analysis
# YOUR CODE HERE

### Exercise 4: Complete EDA

Write a function `quick_eda(df)` that takes a DataFrame and prints:
1. Shape
2. Data types
3. Missing value summary (only columns with missing values)
4. First 5 rows

In [ ]:
def quick_eda(df):
    """Perform quick exploratory data analysis on a DataFrame."""
    # YOUR CODE HERE
    pass